# 第 3 章 — 嵌入（Embeddings）

现在每个词都有了数字，但像 9246 这样的数字只是一个标签。
模型无法从标签中学习，它需要语义——
需要知道 cat 和 dog 相似，而 cat 和 car 不同。

嵌入（embedding）是一组浮点数向量，用来表示词的含义。
可以把它想象成在大空间里给每个词一个坐标：
含义相近的词会靠在一起，含义不同的词会离得很远。

这些嵌入最初是随机数。训练过程中，
模型会学会把相关的词拉近。
训练足够久之后，cat 和 dog 会成为邻居，
而 car 会在空间的另一侧。
本 notebook 将展示这张查表如何工作，
以及学习如何让嵌入变得有意义。

## 导入

In [1]:
import torch
import torch.nn as nn
import math

## 嵌入层

In [2]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embed(x) * math.sqrt(self.d_model)

## 创建嵌入表并测试

In [3]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"嵌入表形状: [{vocab_size}, {d_model}]")
print(f"总参数量: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[12, 45, 678, 2, 890]])
print(f"输入 token ID: {token_ids}")
print(f"输入形状: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"输出形状: {output.shape}")
print(f"每个 token 现在是一个 {d_model}-dimensional vector")
print(f"第一个 token 向量预览: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[ 12,  45, 678,   2, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [-16.388792037963867, -13.410012245178223, 8.839740753173828, 23.09480094909668, -17.038358688354492] ...


## 缩放技巧

我们将嵌入乘以 sqrt(d_model)。
这样在后续加入位置编码之前，数值能保持在合适的尺度。
若不缩放，位置信息会被淹没。

In [4]:
embed_raw = nn.Embedding(vocab_size, d_model)
x = embed_raw(token_ids)

print(f"未缩放: mean={x.mean():.4f}, std={x.std():.4f}")
print(f"已缩放: mean={(x * math.sqrt(d_model)).mean():.4f}, std={(x * math.sqrt(d_model)).std():.4f}")

Without scaling: mean=-0.0146, std=1.0058
With scaling: mean=-0.4043, std=27.8724


## 用分词器查嵌入

现在把它与第 2 章的真实分词器连接起来。

In [5]:
from dataclasses import dataclass
import tiktoken

@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)

    @property
    def vocab_size(self):
        return self.config.vocab_size

tokenizer = SimpleTokenizer()
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text)
print(f"文本: {text}")
print(f"Token ID: {token_ids}")
print()

real_embedding = Embedding(tokenizer.vocab_size, d_model=768)
input_tensor = torch.tensor([token_ids])
vectors = real_embedding(input_tensor)
print(f"输出形状: {vectors.shape}")
print(f"{len(token_ids)} 个 token 各自变成 768 维向量")
print(f"词 cat（token {token_ids[1]}）的向量: {vectors[0, 1, :3].tolist()} ...")

Text: The cat sat on the mat
Token IDs: [464, 3797, 3332, 319, 262, 2603]

Output shape: torch.Size([1, 6, 768])
Each of the 6 tokens is now a 768-dim vector
The word cat (token 3797) has vector: [-43.199928283691406, -31.866304397583008, 17.82335090637207] ...
